# Domain Adaptation: From Domain Shift to Adversarial Feature Alignment
## MNIST → MNIST-M

**The tension in one sentence:** a digit classifier that scores **~99%** on clean MNIST can crater to **~50–60%** the moment the test images are *colorized* — even though they are still, obviously, the same digits.

That gap is called **domain shift**, and closing it is the whole job of **domain adaptation**. This notebook tells the story in four parts:

1. **Reproduce & measure domain shift** — train on clean MNIST (the *source*), watch accuracy collapse on a colorized *target*.
2. **See *why* it fails** — visualize the model's features and find source/target sitting in separate clusters.
3. **Fix it with DANN** — align the two domains adversarially with a Gradient Reversal Layer.
4. **Understand the residual limitation** — why feature alignment alone leaves points near the decision boundary, and what entropy-based methods do about it.

**Demo philosophy:** MNIST-scale, CPU/T4-friendly, fully seeded, and runnable end-to-end (*Run All*) in a single class session.

## Learning objectives

- **LO1** — Reproduce and quantify domain shift: high source accuracy, collapsed target accuracy.
- **LO2** — Visualize *why* a source-only model fails (separated feature clusters) and state the feature-alignment idea.
- **LO3** — Understand and implement **Domain Adversarial Training (DANN)**: a shared feature extractor + label predictor + domain classifier, coupled by a Gradient Reversal Layer under a min–max objective.
- **LO4** — Measure the target-accuracy lift from DANN and confirm, visually, that features now overlap.
- **LO5** — Explain the limitation of pure alignment and how entropy minimization / decision-boundary methods (DIRT-T, MCD) address it.

**Prerequisites:** basic neural-network training, supervised image classification on MNIST, and backpropagation.

## Roadmap

| Section | Cells | Learning objective |
|---|---|---|
| Setup (imports, seeding, device) | C03 | — |
| **Part 1 — Domain shift** (framing → source MNIST → target MNIST-M → viewer → model → train → measure → interpret) | C04–C11 | LO1 |
| **Part 2 — Feature alignment** (framing → source-only feature t-SNE → motivate DANN) | C12–C14 | LO2 |
| **Part 3 — DANN** (framing → Gradient Reversal Layer → DANN module → train) | C15–C18 | LO3 |
| **Part 4 — Results & decision boundary** (accuracy lift → feature overlap → entropy framing → entropy demo → synthesis) | C19–C23 | LO4, LO5 |

In [ ]:
# === Environment setup (run me first) ===
# Ensure ipywidgets is available (numpy/torch/torchvision/matplotlib/scikit-learn ship with Colab).
try:
    import ipywidgets  # noqa: F401
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'])
    import ipywidgets  # noqa: F401

import random
import copy
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

SEED = 42

def set_seed(seed: int = 42) -> None:
    '''Seed python / numpy / torch RNGs so a full Run All is reproducible.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Inline plotting (guarded so the cell also runs in a plain, non-IPython interpreter).
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True

# Smoke checks: reproducible draws + a valid device handle.
set_seed(SEED); _a = torch.rand(3)
set_seed(SEED); _b = torch.rand(3)
assert torch.allclose(_a, _b), 'set_seed should make torch.rand reproducible'
assert isinstance(DEVICE, torch.device) and DEVICE.type in ('cuda', 'cpu')
assert callable(set_seed)
print(f'torch {torch.__version__} | device: {DEVICE} | CPU/T4-friendly demo')

## Part 1 — Domain shift

**Domain shift** is a mismatch between the distribution a model is *trained* on and the one it is *tested* on. We name the two distributions:

- **Source domain** — where we have plenty of *labeled* data (clean MNIST here).
- **Target domain** — where we actually want to perform well, but labels are scarce or absent (a colorized, MNIST-M-style variant).

The digits are identical in *content*; only their *appearance* (color, texture, background) changes. Yet a source-only classifier can drop from **~99.5%** on the source to **~57.5%** on the target — exactly the collapse reported in the lecture. In the next cells we **reproduce that drop** on our own MNIST → MNIST-M setup and measure it.

In [ ]:
# === Source domain: MNIST in 3-channel form ===
# to_rgb_transform: grayscale -> 3x28x28 tensor in ~[-1, 1] so source and target share input shape.
to_rgb_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

NUM_CLASSES = 10
BATCH_SIZE = 128
SUBSET_TRAIN = 6000   # raise toward 60000 for a stronger (slower) baseline
SUBSET_TEST = 2000    # raise toward 10000 for tighter accuracy estimates

try:
    source_train_full = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=to_rgb_transform)
    source_test_full = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=to_rgb_transform)
except Exception:
    print('Failed to download MNIST - check your network / Colab runtime, then re-run this cell.')
    raise

def _subsample(dataset, n):
    set_seed(SEED)
    n = min(n, len(dataset))                      # guard against small installs
    perm = torch.randperm(len(dataset))[:n]
    return torch.utils.data.Subset(dataset, perm.tolist())

source_train_set = _subsample(source_train_full, SUBSET_TRAIN)
source_test_set = _subsample(source_test_full, SUBSET_TEST)

source_train_loader = torch.utils.data.DataLoader(source_train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0)
source_test_loader = torch.utils.data.DataLoader(source_test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

xb, yb = next(iter(source_train_loader))
assert xb.shape[1:] == (3, 28, 28)
assert yb.min() >= 0 and yb.max() <= 9
assert xb.min() >= -1.01 and xb.max() <= 1.01
assert len(source_train_loader) > 0 and len(source_test_loader) > 0
print(f'source train batches: {len(source_train_loader)} | test batches: {len(source_test_loader)}')
print(f'batch shape: {tuple(xb.shape)} | pixel range: [{xb.min():.2f}, {xb.max():.2f}] | labels {yb.min().item()}..{yb.max().item()}')

In [ ]:
# === Target domain: MNIST-M-style colorization, built programmatically from MNIST ===
def colorize(images, strength: float = 0.6, seed=None):
    '''Colorize a normalized batch of grayscale-as-RGB digits.

    images: (B, 3, 28, 28) normalized to ~[-1, 1].
    strength: 0 -> (almost) original grayscale, 1 -> fully colorized.
    seed: if given, draws reproducible colors via a dedicated Generator
          (the global RNG is left untouched).
    Returns a tensor of the same shape/device, renormalized to ~[-1, 1].
    '''
    assert images.dim() == 4 and images.size(1) == 3, f'colorize expects (B,3,H,W), got {tuple(images.shape)}'
    device = images.device
    gray = (images * 0.5 + 0.5).clamp(0, 1)         # denormalize to [0,1]
    B = gray.size(0)
    if seed is not None:
        gen = torch.Generator(device='cpu').manual_seed(int(seed))
        bg = torch.rand(B, 3, 1, 1, generator=gen)
        fg = torch.rand(B, 3, 1, 1, generator=gen)
    else:
        bg = torch.rand(B, 3, 1, 1)
        fg = torch.rand(B, 3, 1, 1)
    bg, fg = bg.to(device), fg.to(device)
    m = gray.mean(dim=1, keepdim=True)              # digit mask in [0,1]
    out = m * fg + (1.0 - m) * bg                   # colored foreground over colored background
    out = strength * out + (1.0 - strength) * gray  # blend toward grayscale by strength
    out = out.clamp(0, 1)
    out = (out - 0.5) / 0.5                          # renormalize to ~[-1,1]
    return out

def _build_target_loader(source_loader, strength, seed, shuffle):
    '''Push every source batch through colorize (on CPU) and wrap as a TensorDataset loader.'''
    imgs, lbls = [], []
    for i, (xb, yb) in enumerate(source_loader):
        xt = colorize(xb.cpu(), strength=strength, seed=seed + i)  # per-batch seed: reproducible, still varied
        imgs.append(xt)
        lbls.append(yb)
    ds = torch.utils.data.TensorDataset(torch.cat(imgs), torch.cat(lbls))
    return torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)

TARGET_STRENGTH = 0.6
set_seed(SEED)
# NOTE: target labels are stored ONLY for evaluation - DANN's label loss never sees them.
target_train_loader = _build_target_loader(source_train_loader, TARGET_STRENGTH, seed=SEED, shuffle=True)
target_test_loader = _build_target_loader(source_test_loader, TARGET_STRENGTH, seed=SEED + 1, shuffle=False)

xs, ys = next(iter(source_test_loader))
xt = colorize(xs)
assert xt.shape == xs.shape and not torch.allclose(xt, xs)               # colorization actually changes the image
assert torch.allclose(colorize(xs, seed=0), colorize(xs, seed=0))        # reproducible for a fixed seed
assert not torch.allclose(colorize(xs, seed=0), colorize(xs, seed=1))    # different across seeds
print(f'target train batches: {len(target_train_loader)} | test batches: {len(target_test_loader)}')
print(f'colorize changed the image: {not torch.allclose(xt, xs)} | output range [{xt.min():.2f}, {xt.max():.2f}]')

In [ ]:
# === Interactive viewer: source digit vs its colorized target counterpart ===
_viewer_imgs, _viewer_lbls = next(iter(source_test_loader))
_viewer_imgs, _viewer_lbls = _viewer_imgs[:32], _viewer_lbls[:32]

def _denormalize(t):
    return (t * 0.5 + 0.5).clamp(0, 1)

def sample_comparison_explorer(strength: float = 0.6, seed: int = 0, n: int = 8) -> None:
    n = min(n, _viewer_imgs.size(0))                       # never over-slice the cached batch
    imgs, lbls = _viewer_imgs[:n], _viewer_lbls[:n]
    src = _denormalize(imgs).cpu()
    tgt = _denormalize(colorize(imgs, strength=strength, seed=seed)).cpu()
    fig, axes = plt.subplots(2, n, figsize=(1.4 * n, 3.0))
    for j in range(n):
        for row, batch in ((0, src), (1, tgt)):
            ax = axes[row, j]
            ax.imshow(batch[j].permute(1, 2, 0).numpy())
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        axes[0, j].set_title(str(lbls[j].item()), fontsize=10)
    axes[0, 0].set_ylabel('source', fontsize=11)
    axes[1, 0].set_ylabel('target', fontsize=11)
    fig.suptitle(f'Source (top) vs colorized target (bottom) - strength={strength:.2f}')
    plt.tight_layout(); plt.show()

# Static default first, so something is always visible even if widgets fail.
sample_comparison_explorer()

try:
    s_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.6, description='strength')
    seed_slider = widgets.IntSlider(min=0, max=20, value=0, description='seed')
    out = widgets.interactive_output(sample_comparison_explorer, {'strength': s_slider, 'seed': seed_slider})
    display(widgets.VBox([s_slider, seed_slider]), out)
except Exception as e:
    print(f'(widgets unavailable: {e}; showing the static default above)')

In [ ]:
# === Shared backbone: FeatureExtractor + LabelPredictor (reused by baseline AND DANN) ===
class FeatureExtractor(nn.Module):
    '''Small CNN mapping a 3x28x28 image to a flat feature vector.'''
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))   # -> (B, 32, 14, 14)
        x = self.pool(F.relu(self.bn2(self.conv2(x))))   # -> (B, 64, 7, 7)
        return x.flatten(1)                               # -> (B, 64*7*7)

# Derive FEATURE_DIM from a dummy forward so the head can never desync from the backbone.
with torch.no_grad():
    FEATURE_DIM = FeatureExtractor()(torch.zeros(1, 3, 28, 28)).shape[1]

class LabelPredictor(nn.Module):
    '''10-way classifier head. Returns raw logits (downstream code applies softmax / CE).'''
    def __init__(self, feature_dim: int = FEATURE_DIM, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, 100),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(100, num_classes),
        )

    def forward(self, f):
        return self.net(f)

_fe = FeatureExtractor(); _f = _fe(torch.zeros(2, 3, 28, 28))
assert _f.shape == (2, FEATURE_DIM)
assert LabelPredictor()(_f).shape == (2, 10)
assert FEATURE_DIM == 64 * 7 * 7
print(f'FeatureExtractor -> FEATURE_DIM={FEATURE_DIM} (== 64*7*7) | LabelPredictor -> 10 logits')

In [ ]:
# === Train the source-only baseline on source MNIST ===
assert len(source_train_loader) > 0

set_seed(SEED)
source_feature_extractor = FeatureExtractor().to(DEVICE)
source_label_predictor = LabelPredictor().to(DEVICE)

SOURCE_EPOCHS = 3  # deliberately small/fast; raise for a stronger baseline
source_optimizer = torch.optim.Adam(
    list(source_feature_extractor.parameters()) + list(source_label_predictor.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, SOURCE_EPOCHS + 1):
    source_feature_extractor.train(); source_label_predictor.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in source_train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        source_optimizer.zero_grad()
        logits = source_label_predictor(source_feature_extractor(xb))
        loss = criterion(logits, yb)
        loss.backward(); source_optimizer.step()
        assert not math.isnan(loss.item())
        running_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)
    print(f'epoch {epoch}/{SOURCE_EPOCHS}  loss={running_loss/total:.4f}  acc={correct/total:.4f}')

source_feature_extractor.eval(); source_label_predictor.eval()
print(f'baseline trained on {total} source images; final-epoch train acc={correct/total:.4f}')

In [ ]:
# === Quantify domain shift: source-only accuracy on source vs target ===
def evaluate(feature_extractor, label_predictor, loader) -> float:
    '''Top-1 accuracy of (feature_extractor -> label_predictor) over loader.
    Single source of truth, reused by C18 (per-epoch) and C19 (final DANN).'''
    feature_extractor.eval(); label_predictor.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds = label_predictor(feature_extractor(xb)).argmax(1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / max(total, 1)

acc_source = evaluate(source_feature_extractor, source_label_predictor, source_test_loader)
acc_target_source_only = evaluate(source_feature_extractor, source_label_predictor, target_test_loader)

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['Source test', 'Target test'], [acc_source, acc_target_source_only], color=['#4C72B0', '#C44E52'])
for b, v in zip(bars, [acc_source, acc_target_source_only]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f'{v:.3f}', ha='center', fontsize=11)
ax.set_ylim(0, 1); ax.set_ylabel('accuracy'); ax.set_title('Source-only model: domain shift')
plt.tight_layout(); plt.show()

assert 0.0 <= acc_source <= 1.0 and 0.0 <= acc_target_source_only <= 1.0
print(f'source acc = {acc_source:.3f} | target acc = {acc_target_source_only:.3f} | gap = {acc_source - acc_target_source_only:.3f}')

## Interpreting the gap

The bar chart makes domain shift concrete: the *same* network is excellent on source digits yet far weaker on target digits that any human reads instantly.

How we fix this depends on **how much we know about the target domain** — think of a spectrum:

- **A little *labeled* target data** → the simplest fix is **fine-tuning** on it. But when that labeled set is small, the model **overfits** it quickly — useful, yet fragile.
- **Lots of *unlabeled* target data** → the richer, more practical setting this notebook targets. With no target labels we cannot fine-tune directly, so instead we make the *features* domain-invariant. That idea is **feature alignment**, and it leads us to DANN.

## Part 2 — Feature alignment

**The basic idea:** if the feature extractor maps source and target images into the **same feature distribution** — i.e. it learns to *ignore the colors* and keep only digit shape — then a label classifier trained on *source* features will also work on *target* features.

So the question becomes diagnostic: **are the source and target features actually aligned** under our source-only model? The next cell extracts features for both domains and projects them to 2D to find out.

In [ ]:
# === Visualize source-only features: source vs target clusters ===
def extract_features(feature_extractor, loaders: dict, n: int = 600):
    '''Collect up to n features per named loader.
    Returns (feats, domain_ids, class_ids) as numpy arrays; domain_ids: 0=source, 1=target.'''
    feature_extractor.eval()
    feats_all, dom_all, cls_all = [], [], []
    for dom_id, (name, loader) in enumerate(loaders.items()):
        gathered = 0
        with torch.no_grad():
            for xb, yb in loader:
                f = feature_extractor(xb.to(DEVICE)).cpu().numpy()
                feats_all.append(f)
                dom_all.append(np.full(len(f), dom_id, dtype=np.int64))
                cls_all.append(yb.numpy())
                gathered += len(f)
                if gathered >= n:
                    break
    return (np.concatenate(feats_all), np.concatenate(dom_all), np.concatenate(cls_all))

def project(feats, method: str):
    mu, sd = feats.mean(0, keepdims=True), feats.std(0, keepdims=True) + 1e-8
    z = (feats - mu) / sd                                  # standardize for stable layouts
    if method == 't-SNE':
        perp = max(5, min(30, len(z) // 4))                # keep perplexity < n_samples/3
        return TSNE(n_components=2, init='pca', perplexity=perp, random_state=SEED).fit_transform(z)
    return PCA(n_components=2, random_state=SEED).fit_transform(z)

def plot_embedding(emb, domain_ids, class_ids, method: str, color_by: str, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))
    ax.grid(False)
    if color_by == 'domain':
        for d, name, c in [(0, 'source', '#4C72B0'), (1, 'target', '#C44E52')]:
            mask = domain_ids == d
            ax.scatter(emb[mask, 0], emb[mask, 1], s=8, alpha=0.5, color=c, label=name)
        ax.legend()
    else:
        sc = ax.scatter(emb[:, 0], emb[:, 1], s=8, alpha=0.6, c=class_ids, cmap='tab10')
        plt.colorbar(sc, ax=ax, ticks=range(10), label='digit')
    ax.set_title(f'{method}, colored by {color_by}')
    ax.set_xticks([]); ax.set_yticks([])
    return ax.figure

_loaders = {'source': source_test_loader, 'target': target_test_loader}
_src_only_feats = extract_features(source_feature_extractor, _loaders)   # cached so C20 reuses identical points
source_only_embedding_figure = None

def source_only_feature_explorer(method: str = 't-SNE', color_by: str = 'domain') -> None:
    global source_only_embedding_figure
    feats, dom, cls = _src_only_feats
    emb = project(feats, method)
    fig, ax = plt.subplots(figsize=(6, 5))
    plot_embedding(emb, dom, cls, method, color_by, ax=ax)
    fig.suptitle('Source-only features (expect SEPARATED domain clusters)')
    plt.tight_layout(); plt.show()
    source_only_embedding_figure = fig

assert _src_only_feats[0].shape[1] == FEATURE_DIM and set(_src_only_feats[1].tolist()) == {0, 1}
source_only_feature_explorer()

try:
    m_dd = widgets.Dropdown(options=['t-SNE', 'PCA'], value='t-SNE', description='method')
    c_dd = widgets.Dropdown(options=['domain', 'class'], value='domain', description='color by')
    out = widgets.interactive_output(source_only_feature_explorer, {'method': m_dd, 'color_by': c_dd})
    display(widgets.HBox([m_dd, c_dd]), out)
except Exception as e:
    print(f'(widgets unavailable: {e}; showing the static default above)')

## Why the separated clusters explain the failure

The projection shows **two separated clusters**: source features land in one region, target features in another. A label classifier trained only on the *source* region has simply **never seen** the part of feature space where target points live — so its predictions there are unreliable. That is the geometric story behind the accuracy drop in C10.

**The fix:** force the feature extractor to produce features whose *domain is indistinguishable*. We add an adversarial **domain classifier** that tries to tell source from target, and we train the feature extractor to **fool** it. That is exactly **DANN**.

## Part 3 — Domain Adversarial Training (DANN)

DANN couples **three** networks that share one backbone:

- **Feature extractor** $G_f$ — turns an image into features.
- **Label predictor** $G_y$ — classifies the digit (trained on *source* labels only).
- **Domain classifier** $G_d$ — tries to tell *source* from *target* apart.

The **min–max objective**: the label predictor and domain classifier minimize their own losses, while the feature extractor minimizes the label loss but **maximizes** the domain loss — so the two domains become indistinguishable in feature space:

$$\min_{G_f, G_y}\ \max_{G_d}\quad \mathcal{L}_y(G_y, G_f)\ -\ \lambda\,\mathcal{L}_d(G_d, G_f)$$

The sign flip is implemented cleanly by a **Gradient Reversal Layer** (GRL): identity on the forward pass, gradient negated (and scaled by $\lambda$) on the backward pass.

**Failure mode to watch:** the feature extractor could trivially fool the domain classifier by collapsing all features to a constant (e.g. always zero). The **label loss** prevents this — collapsed features cannot classify digits — so the two objectives balance.

*References:* Ganin & Lempitsky, *Unsupervised Domain Adaptation by Backpropagation* (ICML 2015); Ajakan et al., *Domain-Adversarial Training of Neural Networks* (JMLR 2016).

In [ ]:
# === Gradient Reversal Layer (GRL): identity forward, negated + scaled gradient backward ===
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = float(lambda_)
        return x.view_as(x)               # fresh node that records the custom backward; values unchanged

    @staticmethod
    def backward(ctx, grad_output):
        # Negate and scale the upstream gradient; None matches the non-tensor lambda_ input.
        return grad_output.neg() * ctx.lambda_, None

def grad_reverse(x, lambda_: float = 1.0):
    return GradientReversalFunction.apply(x, lambda_)

class GradientReversal(nn.Module):
    '''nn.Module wrapper around grad_reverse, for naming consistency with C17.'''
    def __init__(self, lambda_: float = 1.0):
        super().__init__()
        self.lambda_ = lambda_
    def forward(self, x):
        return grad_reverse(x, self.lambda_)

# Smoke checks: identity forward, gradient negated & scaled by lambda, zero at lambda=0.
_x = torch.randn(4, 3, requires_grad=True)
_y = grad_reverse(_x, 2.0)
assert torch.allclose(_y, _x)
_y.sum().backward()
assert torch.allclose(_x.grad, -2.0 * torch.ones_like(_x))
_x2 = torch.randn(4, 3, requires_grad=True)
grad_reverse(_x2, 0.0).sum().backward()
assert torch.allclose(_x2.grad, torch.zeros_like(_x2))
print('GRL OK: forward is identity; backward returns -lambda * grad (0 at lambda=0).')

In [ ]:
# === Assemble DANN: shared backbone + label branch + adversarial domain branch ===
class DomainClassifier(nn.Module):
    '''2-way domain discriminator (source=0 / target=1).'''
    def __init__(self, feature_dim: int = FEATURE_DIM, hidden: int = 100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden, 2),
        )
    def forward(self, f):
        return self.net(f)

class DANN(nn.Module):
    '''feature_extractor -> label_predictor (label branch);
       feature_extractor -> grad_reverse -> domain_classifier (domain branch).
    Attribute names are fixed by contract so C18/C19/C20 can reach the trained submodules.'''
    def __init__(self, feature_extractor, label_predictor, domain_classifier):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.label_predictor = label_predictor
        self.domain_classifier = domain_classifier

    def forward(self, x, lambda_: float = 1.0):
        f = self.feature_extractor(x)
        class_logits = self.label_predictor(f)
        domain_logits = self.domain_classifier(grad_reverse(f, lambda_))
        return class_logits, domain_logits

_m = DANN(FeatureExtractor(), LabelPredictor(), DomainClassifier())
_cl, _dl = _m(torch.zeros(2, 3, 28, 28), 1.0)
assert _cl.shape == (2, 10) and _dl.shape == (2, 2)
assert hasattr(_m, 'feature_extractor') and hasattr(_m, 'label_predictor') and hasattr(_m, 'domain_classifier')
print('DANN OK: forward(x, lambda) -> (class_logits[B,10], domain_logits[B,2]).')

In [ ]:
# === Train DANN (unsupervised domain adaptation) ===
def lambda_schedule(p: float, gamma: float = 10.0, max_lambda: float = 1.0) -> float:
    '''GRL strength ramp: 0 at p=0, -> max_lambda as p->1 (standard DANN schedule).'''
    return max_lambda * (2.0 / (1.0 + math.exp(-gamma * p)) - 1.0)

def train_dann(max_lambda: float = 1.0, epochs: int = 5, verbose: bool = True):
    set_seed(SEED)
    model = DANN(FeatureExtractor(), LabelPredictor(), DomainClassifier()).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    ce = nn.CrossEntropyLoss()
    steps_per_epoch = min(len(source_train_loader), len(target_train_loader))
    total_steps = epochs * max(steps_per_epoch, 1)
    global_step = 0
    for epoch in range(1, epochs + 1):
        model.train()
        last_label, last_domain, last_lam = 0.0, 0.0, 0.0
        for (xs, ys), (xt, _) in zip(source_train_loader, target_train_loader):
            xs, ys, xt = xs.to(DEVICE), ys.to(DEVICE), xt.to(DEVICE)   # target labels discarded (xt, _)
            p = global_step / max(total_steps, 1)
            lam = lambda_schedule(p, max_lambda=max_lambda)
            optimizer.zero_grad()
            class_logits, src_dom = model(xs, lam)                     # label loss on SOURCE only
            label_loss = ce(class_logits, ys)
            _, tgt_dom = model(xt, lam)
            domain_logits = torch.cat([src_dom, tgt_dom], dim=0)
            domain_labels = torch.cat([torch.zeros(xs.size(0)), torch.ones(xt.size(0))]).long().to(DEVICE)
            domain_loss = ce(domain_logits, domain_labels)
            loss = label_loss + domain_loss
            loss.backward(); optimizer.step()
            assert math.isfinite(loss.item())
            last_label, last_domain, last_lam = label_loss.item(), domain_loss.item(), lam
            global_step += 1
        target_acc = evaluate(model.feature_extractor, model.label_predictor, target_test_loader)
        if verbose:
            print(f'epoch {epoch}/{epochs}  label_loss={last_label:.4f}  domain_loss={last_domain:.4f}  lambda={last_lam:.3f}  target_acc={target_acc:.4f}')
    return model

# Schedule sanity: 0 at p=0, ~max_lambda at p=1, monotonic in between.
assert lambda_schedule(0.0) == 0.0 and abs(lambda_schedule(1.0, max_lambda=1.0) - 1.0) < 1e-3
assert lambda_schedule(0.3) < lambda_schedule(0.6) < lambda_schedule(0.9)

DANN_EPOCHS = 5
DANN_MAX_LAMBDA = 1.0
dann_model = train_dann(DANN_MAX_LAMBDA, DANN_EPOCHS)

# Optional: retrain interactively. interact_manual is button-triggered, so it will NOT rerun on every tick.
try:
    lam_slider = widgets.FloatSlider(min=0.0, max=2.0, step=0.1, value=1.0, description='max lambda')
    ep_slider = widgets.IntSlider(min=1, max=10, value=5, description='epochs')
    @widgets.interact_manual(max_lambda=lam_slider, epochs=ep_slider)
    def _retrain(max_lambda=1.0, epochs=5):
        global dann_model
        dann_model = train_dann(max_lambda, epochs)
        acc = evaluate(dann_model.feature_extractor, dann_model.label_predictor, target_test_loader)
        print(f'retrained DANN -> target accuracy = {acc:.4f}')
except Exception as e:
    print(f'(widgets unavailable: {e}; trained once with defaults above)')

In [ ]:
# === Headline adaptation lift: source-only vs DANN ===
acc_target_dann = evaluate(dann_model.feature_extractor, dann_model.label_predictor, target_test_loader)

groups = ['Source domain', 'Target domain']
source_only_vals = [acc_source, acc_target_source_only]
dann_vals = [acc_source, acc_target_dann]   # DANN source bar reuses acc_source for visual parity
x = np.arange(len(groups)); w = 0.35

fig, ax = plt.subplots(figsize=(6, 4.5))
b1 = ax.bar(x - w / 2, source_only_vals, w, label='Source-only', color='#C44E52')
b2 = ax.bar(x + w / 2, dann_vals, w, label='DANN', color='#55A868')
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.02, f'{b.get_height():.3f}', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(groups); ax.set_ylim(0, 1)
ax.set_ylabel('accuracy'); ax.set_title('Domain adaptation lift (source-only vs DANN)')
ax.legend(); plt.tight_layout(); plt.show()

assert 0.0 <= acc_target_dann <= 1.0
improvement = acc_target_dann - acc_target_source_only
print(f'target accuracy: source-only {acc_target_source_only:.3f} -> DANN {acc_target_dann:.3f} (delta = {improvement:+.3f})')
if improvement <= 0:
    print('Note: with the scaled-down classroom budget DANN can occasionally not improve; raise epochs/subset for a clearer lift.')

In [ ]:
# === Before/after: source-only vs DANN feature alignment ===
_dann_feats = extract_features(dann_model.feature_extractor, {'source': source_test_loader, 'target': target_test_loader})

# Reuse the C13 source-only sample so the 'before' panel is identical; recompute only if missing.
try:
    _src_only_feats
except NameError:
    _src_only_feats = extract_features(source_feature_extractor, {'source': source_test_loader, 'target': target_test_loader})

dann_embedding_figure = None

def dann_feature_explorer(method: str = 't-SNE', color_by: str = 'domain') -> None:
    global dann_embedding_figure
    sfeats, sdom, scls = _src_only_feats
    dfeats, ddom, dcls = _dann_feats
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    plot_embedding(project(sfeats, method), sdom, scls, method, color_by, ax=axes[0])   # before (source-only)
    axes[0].set_title(f'source-only ({method}, {color_by})')
    plot_embedding(project(dfeats, method), ddom, dcls, method, color_by, ax=axes[1])   # after (DANN)
    axes[1].set_title(f'DANN ({method}, {color_by})')
    fig.suptitle('Feature alignment: source-only (left) vs DANN (right)')
    plt.tight_layout(); plt.show()
    dann_embedding_figure = fig

assert _dann_feats[0].shape[1] == FEATURE_DIM
dann_feature_explorer()

try:
    m_dd2 = widgets.Dropdown(options=['t-SNE', 'PCA'], value='t-SNE', description='method')
    c_dd2 = widgets.Dropdown(options=['domain', 'class'], value='domain', description='color by')
    out = widgets.interactive_output(dann_feature_explorer, {'method': m_dd2, 'color_by': c_dd2})
    display(widgets.HBox([m_dd2, c_dd2]), out)
except Exception as e:
    print(f'(widgets unavailable: {e}; showing the static default above)')

## Part 4 — Considering the decision boundary

DANN aligns the *marginal* feature distributions — but alignment alone has a blind spot. Aligned target points can still land **right next to the source decision boundary**, where the classifier is **uncertain** and a small perturbation flips the prediction.

**Entropy minimization** addresses this: encourage **confident** (low-entropy) predictions on target data, which pushes target points *away* from the boundary into the interior of a class region. Methods built on this idea include **DIRT-T** (arXiv:1802.08735) and **Maximum Classifier Discrepancy / MCD** (arXiv:1712.02560).

The next cell *diagnoses* this on our trained DANN by looking at the **entropy** of its target predictions — how many target points sit in the uncertain, near-boundary region.

In [ ]:
# === Decision-boundary / entropy diagnostic on the trained DANN ===
dann_model.eval()
_ent_list = []
with torch.no_grad():
    for xt, _ in target_test_loader:
        probs = F.softmax(dann_model.label_predictor(dann_model.feature_extractor(xt.to(DEVICE))), dim=1)
        ent = -(probs * (probs + 1e-12).log()).sum(1)        # +eps inside log avoids NaN
        _ent_list.append(ent.cpu())
target_entropies = torch.cat(_ent_list)
MAX_ENTROPY = math.log(NUM_CLASSES)                          # uniform-prediction upper bound
assert target_entropies.min() >= 0.0 and target_entropies.max() <= MAX_ENTROPY + 1e-4

def entropy_explorer(threshold: float) -> None:
    threshold = float(min(max(threshold, 0.0), MAX_ENTROPY))
    frac_confident = float((target_entropies <= threshold).float().mean())
    frac_uncertain = 1.0 - frac_confident
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(target_entropies.numpy(), bins=40, color='#4C72B0', alpha=0.85)
    ax.axvline(threshold, color='k', linestyle='--')
    ax.axvspan(threshold, MAX_ENTROPY, color='#C44E52', alpha=0.15, label='near boundary (uncertain)')
    ax.set_xlabel('prediction entropy (nats)'); ax.set_ylabel('count')
    ax.set_title(f'Target entropy: confident {frac_confident*100:.1f}% | uncertain {frac_uncertain*100:.1f}% (thr={threshold:.2f})')
    ax.legend(); plt.tight_layout(); plt.show()

entropy_explorer(MAX_ENTROPY / 2)

try:
    thr_slider = widgets.FloatSlider(min=0.0, max=MAX_ENTROPY, step=0.05, value=MAX_ENTROPY / 2, description='threshold')
    out = widgets.interactive_output(entropy_explorer, {'threshold': thr_slider})
    display(thr_slider, out)
except Exception as e:
    print(f'(widgets unavailable: {e}; showing the static default above)')

## Synthesis & outlook

**The four-part story, recapped:**

1. **Domain shift is real** (C10) — the source-only model scored high on source digits and dropped sharply on colorized target digits.
2. **We saw why** (C13) — its features form *separated* source/target clusters, so the source-trained classifier meets target points in unseen territory.
3. **DANN closes the gap** (C19–C20) — adversarial alignment via the Gradient Reversal Layer lifted target accuracy and **overlapped** the feature clusters.
4. **Alignment is not the whole story** (C22) — many target points still sit near the decision boundary; entropy-based methods push them into confident regions.

**Intentionally deferred (great extensions):**

- The full spectrum of problem settings — **open-set / partial / universal** domain adaptation.
- **Test-Time Training (TTT)** — adapting at inference time (arXiv:1909.13231).
- **Domain generalization** — *zero* target data at training time (arXiv:2003.13216).

**Exercises:**

1. Raise `TARGET_STRENGTH` in C06 and re-run: how do the source-only gap and DANN's recovery change?
2. Sweep `max_lambda` in C18 — find where too-strong adversarial pressure starts to hurt the label loss.
3. Add an **entropy-minimization** term to DANN's target loss and re-check the C22 histogram and target accuracy.

*References:* DANN (Ganin & Lempitsky 2015; Ajakan et al. 2016); DIRT-T (1802.08735); Maximum Classifier Discrepancy (1712.02560); Test-Time Training (1909.13231); Domain Generalization (2003.13216).